In [169]:
#import libraries
import pandas as pd
import pickle
import nltk

pd.set_option('display.max_columns', None)

In [170]:
#loading the data
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

In [171]:
print("Movies CSV:")
display(movies.head(5))
print("\nRatings CSV:")
display(ratings.head(5))

Movies CSV:


,id,title,genres,overview,cast,director,release_date,popularity,vote_average,vote_count,poster_url
0,1523145,Your Heart Will Be Broken,"Romance, Drama",High school student Polina is saved from bully...,"Daniel Vegas, Veronika Zhuravleva, Ivan Trushi...",Mikhail Vaynberg,2026-03-26,1043.4595,7.000,21,https://image.tmdb.org/t/p/w500/7wIBfBl2gejt6x...
1,83533,Avatar: Fire and Ash,"Science Fiction, Adventure, Fantasy",In the wake of the devastating war against the...,"Sam Worthington, Zoe Saldaña, Sigourney Weaver...",James Cameron,2025-12-17,369.7046,7.276,2019,https://image.tmdb.org/t/p/w500/bRBeSHfGHwkEpI...
2,1084187,Pretty Lethal,"Action, Thriller",A troupe of ballerinas find themselves fightin...,"Maddie Ziegler, Lana Condor, Iris Apatow, Mill...",Vicky Jewson,2026-03-13,304.1287,6.777,146,https://image.tmdb.org/t/p/w500/znTPnXCK3lEQJg...
3,1115544,Mike & Nick & Nick & Alice,"Comedy, Science Fiction, Crime",Two gangsters and the woman they love try to s...,"James Marsden, Vince Vaughn, Eiza González, Ke...",BenDavid Grabinski,2026-03-14,323.4594,6.752,101,https://image.tmdb.org/t/p/w500/7F0jc75HrSkLVc...
4,1290821,Shelter,"Action, Crime, Thriller",A man living in self-imposed exile on a remote...,"Jason Statham, Bodhi Rae Breathnach, Michael S...",Ric Roman Waugh,2026-01-28,284.1759,6.729,435,https://image.tmdb.org/t/p/w500/buPFnHZ3xQy6vZ...



Ratings CSV:


,userId,movieId,rating
0,1,1208348,7.9
1,1,1193501,6.0
2,1,561362,5.6
3,1,434853,5.6
4,1,10867,7.4


In [172]:
#drop movies with missing essential info
movies.dropna(subset=['title','genres','overview'], inplace=True)


In [173]:
#fill missing values
movies['cast']= movies['cast'].fillna('Unknown')
movies['director']= movies['director'].fillna('Unknown')
movies['release_date']= movies['release_date'].fillna('Unknown')
movies['poster_url']= movies['poster_url'].fillna('Unknown')


In [174]:
#remove duplicates
movies.drop_duplicates(subset='id', inplace=True)

In [175]:
#convert text to lowercase
movies['genres'] = movies['genres'].str.lower()
movies['overview'] = movies['overview'].str.lower()
movies['cast'] = movies['cast'].str.lower()
movies['director'] = movies['director'].str.lower()

In [176]:
print("Movies cleaned:",movies.shape)
display(movies.head())

Movies cleaned: (955, 11)


,id,title,genres,overview,cast,director,release_date,popularity,vote_average,vote_count,poster_url
0,1523145,Your Heart Will Be Broken,"romance, drama",high school student polina is saved from bully...,"daniel vegas, veronika zhuravleva, ivan trushi...",mikhail vaynberg,2026-03-26,1043.4595,7.000,21,https://image.tmdb.org/t/p/w500/7wIBfBl2gejt6x...
1,83533,Avatar: Fire and Ash,"science fiction, adventure, fantasy",in the wake of the devastating war against the...,"sam worthington, zoe saldaña, sigourney weaver...",james cameron,2025-12-17,369.7046,7.276,2019,https://image.tmdb.org/t/p/w500/bRBeSHfGHwkEpI...
2,1084187,Pretty Lethal,"action, thriller",a troupe of ballerinas find themselves fightin...,"maddie ziegler, lana condor, iris apatow, mill...",vicky jewson,2026-03-13,304.1287,6.777,146,https://image.tmdb.org/t/p/w500/znTPnXCK3lEQJg...
3,1115544,Mike & Nick & Nick & Alice,"comedy, science fiction, crime",two gangsters and the woman they love try to s...,"james marsden, vince vaughn, eiza gonzález, ke...",bendavid grabinski,2026-03-14,323.4594,6.752,101,https://image.tmdb.org/t/p/w500/7F0jc75HrSkLVc...
4,1290821,Shelter,"action, crime, thriller",a man living in self-imposed exile on a remote...,"jason statham, bodhi rae breathnach, michael s...",ric roman waugh,2026-01-28,284.1759,6.729,435,https://image.tmdb.org/t/p/w500/buPFnHZ3xQy6vZ...


In [177]:
#drop missing values
ratings.dropna(subset=['userId','movieId','rating'], inplace=True)

In [178]:
#convert ids to integer
ratings['userId']= ratings['userId'].astype(int)
ratings['movieId']= ratings['movieId'].astype(int)

In [179]:
#remove duplicates
ratings.drop_duplicates(subset=['userId','movieId'], inplace=True)

In [180]:
#clip ratings to valid range (0-10)
ratings['rating']= ratings['rating'].clip(0,10)

In [181]:
print("Ratings Cleaned:", ratings.shape)
display(ratings.head())

Ratings Cleaned: (3574, 3)


,userId,movieId,rating
0,1,1208348,7.9
1,1,1193501,6.0
2,1,561362,5.6
3,1,434853,5.6
4,1,10867,7.4


In [182]:
# Combine features into tags
movies['tags'] = (movies['genres'].fillna('') + ' ' +
                  movies['overview'].fillna('') + ' ' +
                  movies['cast'].fillna('') + ' ' +
                  movies['director'].fillna(''))

new_df = movies[['id', 'title', 'tags', 'vote_average', 'vote_count',
                 'poster_url', 'release_date']]

In [183]:
new_df['pop_score']= new_df['vote_average']* new_df['vote_count']

In [184]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
new_df['tags'] = new_df['tags'].str.lower()

In [185]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
def stem(text):
    return " ".join(ps.stem(word) for word in text.split())
new_df['tags'] = new_df['tags'].apply(stem)

In [186]:
#content-based similarity\
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
content_similarity = cosine_similarity(vectors)

In [187]:
def content_recommend(movie):
    idx = new_df[new_df['title'] == movie].index[0]
    top = sorted(list(enumerate(content_similarity[idx])),reverse=True, key=lambda x: x[1])[1:7]
    return new_df.iloc[[i[0] for i in top]]['title'].tolist()  # Fixed

In [188]:
content_recommend('Avatar: Fire and Ash')


['Avatar: The Way of Water',
 'Avatar',
 'Guardians of the Galaxy Vol. 2',
 'Guardians of the Galaxy Vol. 3',
 'Black Panther: Wakanda Forever',
 'X-Men: Days of Future Past']

In [189]:
import pickle

# Save processed movie dataframe
pickle.dump(new_df, open("movies.pkl", "wb"))

# Save similarity matrix
pickle.dump(content_similarity, open("content_sim.pkl", "wb"))

print("Pickle files saved successfully!")

Pickle files saved successfully!


In [191]:
#user-movie matrix
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating',
).fillna(0)
print(user_movie_matrix.shape)

(100, 952)
